<a href="https://colab.research.google.com/github/LionPaul/retro-game-miner/blob/main/snes/notebooks/04_filter_scored.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os

# ==============================================================================
# PIPELINE CONFIGURATION
# ==============================================================================
CONSOLE_NAME = "snes"

DATASETS_DIR = "datasets"
# ADJUSTMENT: Correct chained sequence tracking for Step 04
INPUT_FILENAME = f"3_{CONSOLE_NAME}_games_populated.xlsx"     # Reading from Code 3
OUTPUT_FILENAME = f"4_{CONSOLE_NAME}_games_scored_only.xlsx"   # Generating base number 4

INPUT_PATH = os.path.join(DATASETS_DIR, INPUT_FILENAME)
OUTPUT_PATH = os.path.join(DATASETS_DIR, OUTPUT_FILENAME)
# ==============================================================================

def pipeline_audit_and_filter(input_path, output_path):
    """
    Filters out records without scores to optimize downstream API data costs.
    Acts as the Gold Filtered Layer of the Medallion Architecture.
    """
    print(f"✂️ [CODE 04] Starting Data Audit and Filtration for console: {CONSOLE_NAME.upper()}")

    # 1. Pipeline gatekeeper check
    if not os.path.exists(input_path):
        print(f"❌ Critical Error: Input file '{input_path}' not found. Please run Notebook 03 first.")
        return None

    # 2. Loading populated data
    df_raw = pd.read_excel(input_path)
    total_raw_rows = len(df_raw)
    print(f"📊 Dataset loaded successfully. Total records on current stream: {total_raw_rows}")

    # 3. Data Audit Metrics (Pipeline KPIs)
    games_with_score = df_raw['IGDB_Score'].notna().sum()
    games_without_score = df_raw['IGDB_Score'].isna().sum()
    api_success_rate = round((games_with_score / total_raw_rows) * 100, 2)

    print("\n--- DATASET AUDIT INFORMATION ---")
    print(f"✅ Titles with localized Score: {games_with_score}")
    print(f"⚠️ Titles without Score (Non-Analyzable): {games_without_score}")
    print(f"📈 API Search Efficiency Rate: {api_success_rate}%")
    print("---------------------------------\n")

    print("⚡ Expurgating non-analyzable records and optimizing dataset footprint...")

    # 4. Surgical Filtration Process
    # Drop rows where IGDB_Score is null
    df_filtered = df_raw.dropna(subset=['IGDB_Score']).copy()

    # Drop accidental duplicates based on the clean search key to ensure unicity for Code 5
    df_filtered = df_filtered.drop_duplicates(subset=['T1_Clean'])

    total_final_rows = len(df_filtered)
    removed_rows = total_raw_rows - total_final_rows

    # 5. Optimized Artifact Persistence
    df_filtered.to_excel(output_path, index=False)

    print(f"📉 Filtration complete. {removed_rows} redundant or unmapped rows were cleaned.")
    print(f"🎯 Total unique keys ready for Category mapping (Code 05): {total_final_rows}")
    print(f"💾 Optimized dataset saved at: {output_path}")
    print("═" * 80)

    # Returns preview for Colab inspector panel
    return df_filtered[['Title', 'T1_Clean', 'IGDB_Score']].head(5)

# Trigger the optimization process
df_scored_preview = pipeline_audit_and_filter(INPUT_PATH, OUTPUT_PATH)